In [1]:
# =============================================================================
# Arabic Sentiment Analysis — BiLSTM + Multi-Head Attention (Fixed)
# =============================================================================

import warnings, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

TRAIN_PATH    = "/content/train.csv"
VAL_PATH      = "/content/val.csv"
TEST_PATH     = "/content/test.csv"
TEXT_COL      = "clean_text"
LABEL_COL     = "labels"

TOKENIZER_ID  = "aubmindlab/bert-base-arabertv2"
MAX_SEQ_LEN   = 128
NUM_LABELS    = 2

# Model hyperparameters
EMBED_DIM       = 300
LSTM_HIDDEN = 128
LSTM_LAYERS = 1
ATT_HEADS = 4
ATT_HEAD_DIM = 32
MLP_HIDDEN = 128
DROPOUT = 0.5

# Training
BATCH_SIZE    = 64
EPOCHS        = 4
LR            = 5e-4
WEIGHT_DECAY  = 1e-4
GRAD_CLIP     = 5.0
PATIENCE      = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device : {DEVICE}")


from transformers import AutoTokenizer
tokenizer  = AutoTokenizer.from_pretrained(TOKENIZER_ID)
VOCAB_SIZE = tokenizer.vocab_size
PAD_ID     = tokenizer.pad_token_id
print(f"  Tokenizer vocab : {VOCAB_SIZE:,} | PAD id : {PAD_ID}\n")


class ArabicSentimentDataset(Dataset):
    def __init__(self, df, max_len=MAX_SEQ_LEN):
        self.texts  = df[TEXT_COL].fillna("").astype(str).tolist()
        self.labels = df[LABEL_COL].astype(int).tolist()
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
            return_token_type_ids=False,
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long),
        }


def make_loader(df, shuffle):
    return DataLoader(
        ArabicSentimentDataset(df),
        batch_size=BATCH_SIZE, shuffle=shuffle,
        num_workers=0, pin_memory=DEVICE.type == "cuda"
    )


class HighwayLayer(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.transform = nn.Linear(dim, dim)
        self.gate      = nn.Linear(dim, dim)
        nn.init.constant_(self.gate.bias, -1.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        T = torch.sigmoid(self.gate(x))
        H = F.relu(self.transform(x))
        return T * H + (1.0 - T) * x


class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, input_dim: int, num_heads: int, head_dim: int):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = head_dim

        self.W = nn.Linear(input_dim, num_heads * head_dim)
        self.v = nn.Parameter(torch.randn(num_heads, head_dim))
        self.out_proj = nn.Linear(num_heads * head_dim, input_dim)
        self.dropout  = nn.Dropout(DROPOUT)
        nn.init.xavier_uniform_(self.v.unsqueeze(0))

    def forward(self, hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        B, T, D = hidden.shape

        # Project to all heads
        proj = torch.tanh(self.W(hidden)).view(B, T, self.num_heads, self.head_dim)  # B, T, H, d

        # Compute attention scores
        scores = torch.einsum("hd,bthd->bth", self.v, proj)  # B, T, H
        scores = scores.permute(0, 2, 1)                      # B, H, T

        # Mask padding positions
        mask_exp = mask.unsqueeze(1).expand_as(scores)       # B, H, T
        scores = scores.masked_fill(mask_exp == 0, -1e9)
        alpha  = F.softmax(scores, dim=-1)                  # B, H, T
        alpha  = self.dropout(alpha)

        # Weighted sum per head: B, H, T × B, T, H, d → B, H, d
        proj = proj.permute(0, 2, 1, 3)                     # B, H, T, d
        ctx_heads = torch.einsum("bht,bhtd->bhd", alpha, proj)  # B, H, d

        # Flatten heads and project
        ctx_flat = ctx_heads.reshape(B, self.num_heads * self.head_dim)  # B, H*d
        out = self.out_proj(ctx_flat)
        return self.dropout(out)


class BiLSTMAttentionClassifier(nn.Module):
    def __init__(self, vocab_size: int, num_labels: int = NUM_LABELS):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=PAD_ID)
        nn.init.uniform_(self.embedding.weight, -0.05, 0.05)
        with torch.no_grad():
            self.embedding.weight[PAD_ID].fill_(0.0)

        self.highway = HighwayLayer(EMBED_DIM)
        self.embed_dropout = nn.Dropout(DROPOUT)

        self.bilstm = nn.LSTM(
            input_size    = EMBED_DIM,
            hidden_size   = LSTM_HIDDEN,
            num_layers    = LSTM_LAYERS,
            bidirectional = True,
            batch_first   = True,
            dropout       = LSTM_DROPOUT if LSTM_LAYERS > 1 else 0.0,
        )
        lstm_out_dim = LSTM_HIDDEN * 2
        self.lstm_norm = nn.LayerNorm(lstm_out_dim)

        self.attention = MultiHeadAttentionPooling(
            input_dim = lstm_out_dim,
            num_heads = ATT_HEADS,
            head_dim  = ATT_HEAD_DIM,
        )

        self.residual_weight = nn.Parameter(torch.tensor(0.3))

        self.classifier = nn.Sequential(
            nn.Linear(lstm_out_dim, MLP_HIDDEN),
            nn.LayerNorm(MLP_HIDDEN),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(MLP_HIDDEN, MLP_HIDDEN // 2),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(MLP_HIDDEN // 2, num_labels),
        )

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor):
        emb = self.embedding(input_ids)
        emb = self.highway(emb)
        emb = self.embed_dropout(emb)

        lengths = attention_mask.sum(dim=1).cpu()
        packed  = nn.utils.rnn.pack_padded_sequence(
            emb, lengths, batch_first=True, enforce_sorted=False
        )
        lstm_out, _ = self.bilstm(packed)
        lstm_out, _ = nn.utils.rnn.pad_packed_sequence(
            lstm_out, batch_first=True, total_length=emb.size(1)
        )
        lstm_out = self.lstm_norm(lstm_out)

        attn_ctx = self.attention(lstm_out, attention_mask)

        mask_exp = attention_mask.unsqueeze(-1).float()
        masked   = lstm_out * mask_exp + (1 - mask_exp) * (-1e9)
        max_pool = masked.max(dim=1).values

        w      = torch.sigmoid(self.residual_weight)
        pooled = w * attn_ctx + (1.0 - w) * max_pool

        return self.classifier(pooled)


print("=" * 65); print("  📂  Loading Data"); print("=" * 65)
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)
print(f"  Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}")
print(f"\n  Labels:\n{train_df[LABEL_COL].value_counts()}\n")

train_loader = make_loader(train_df, shuffle=True)
val_loader   = make_loader(val_df,   shuffle=False)
test_loader  = make_loader(test_df,  shuffle=False)


model = BiLSTMAttentionClassifier(vocab_size=VOCAB_SIZE).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Trainable parameters : {total_params:,}\n")

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr         = LR,
    steps_per_epoch= len(train_loader),
    epochs         = EPOCHS,
    pct_start      = 0.2,
    anneal_strategy= "cos",
)

counts    = train_df[LABEL_COL].value_counts().sort_index()
weights   = torch.tensor([len(train_df) / (NUM_LABELS * c) for c in counts.values],
                         dtype=torch.float).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

def train_one_epoch(epoch):
    model.train()
    total_loss, preds_all, labels_all = 0.0, [], []
    for step, batch in enumerate(train_loader):
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        y    = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        logits = model(ids, mask)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        total_loss += loss.item()
        preds_all.extend(logits.argmax(-1).cpu().numpy())
        labels_all.extend(y.cpu().numpy())
        if (step + 1) % 100 == 0:
            print(f"    E{epoch+1} step {step+1}/{len(train_loader)} "
                  f"loss {loss.item():.4f} lr {scheduler.get_last_lr()[0]:.2e}")
    return (total_loss / len(train_loader),
            accuracy_score(labels_all, preds_all),
            f1_score(labels_all, preds_all, average="macro", zero_division=0))


@torch.no_grad()
def eval_loader(loader):
    model.eval()
    total_loss, preds_all, labels_all = 0.0, [], []
    for batch in loader:
        ids   = batch["input_ids"].to(DEVICE)
        mask  = batch["attention_mask"].to(DEVICE)
        y     = batch["label"].to(DEVICE)
        logits = model(ids, mask)
        total_loss += criterion(logits, y).item()
        preds_all.extend(logits.argmax(-1).cpu().numpy())
        labels_all.extend(y.cpu().numpy())
    return total_loss / len(loader), np.array(preds_all), np.array(labels_all)


def print_metrics(title, y_true, y_pred):
    names = ["Negative", "Positive"]
    print(f"\n{'─'*65}\n  {title}\n{'─'*65}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=names, zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")


print("=" * 65); print("  🚀  Training BiLSTM + Attention from Scratch"); print("=" * 65)

best_val_f1, best_state = 0.0, None
patience_counter        = 0
history                 = []

for epoch in range(EPOCHS):
    print(f"\n{'='*65}\n  EPOCH {epoch+1}/{EPOCHS}\n{'='*65}")
    tr_loss, tr_acc, tr_f1 = train_one_epoch(epoch)
    vl_loss, vl_preds, vl_labels = eval_loader(val_loader)
    vl_f1 = f1_score(vl_labels, vl_preds, average="macro", zero_division=0)
    history.append({"epoch": epoch+1, "train_loss": tr_loss, "train_f1": tr_f1,
                    "val_loss": vl_loss, "val_f1": vl_f1})
    print(f"\n  Train Loss {tr_loss:.4f} | Acc {tr_acc:.4f} | F1 {tr_f1:.4f}")
    print(f"  Val   Loss {vl_loss:.4f} | F1 {vl_f1:.4f}")

    if vl_f1 > best_val_f1:
        best_val_f1      = vl_f1
        best_state       = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        torch.save(best_state, "best_bilstm_attention.pt")
        print(f"  ✅ Best model saved (val F1 = {best_val_f1:.4f})")
    else:
        patience_counter += 1
        print(f"  ⏳ No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print(f"\n  Early stopping triggered after epoch {epoch+1}.")
            break


model.load_state_dict(best_state)
_, vp, vl = eval_loader(val_loader);  print_metrics("Validation Set", vl, vp)
_, tp, tl = eval_loader(test_loader); print_metrics("Test Set",       tl, tp)

print("\n" + pd.DataFrame(history).to_string(index=False, float_format="{:.4f}".format))
print("\n  Model saved to: best_bilstm_attention.pt")

✅ Device : cuda


config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

  Tokenizer vocab : 64,000 | PAD id : 31

  📂  Loading Data
  Train 5,068 | Val 1,086 | Test 1,086

  Labels:
labels
1    2576
0    2492
Name: count, dtype: int64

  Trainable parameters : 19,929,019

  🚀  Training BiLSTM + Attention from Scratch

  EPOCH 1/4

  Train Loss 0.6995 | Acc 0.5057 | F1 0.5029
  Val   Loss 0.6914 | F1 0.3378
  ✅ Best model saved (val F1 = 0.3378)

  EPOCH 2/4

  Train Loss 0.6948 | Acc 0.5132 | F1 0.5131
  Val   Loss 0.6880 | F1 0.4491
  ✅ Best model saved (val F1 = 0.4491)

  EPOCH 3/4

  Train Loss 0.6655 | Acc 0.5998 | F1 0.5997
  Val   Loss 0.5926 | F1 0.6761
  ✅ Best model saved (val F1 = 0.6761)

  EPOCH 4/4

  Train Loss 0.5082 | Acc 0.7908 | F1 0.7908
  Val   Loss 0.5137 | F1 0.7678
  ✅ Best model saved (val F1 = 0.7678)

─────────────────────────────────────────────────────────────────
  Validation Set
─────────────────────────────────────────────────────────────────
  Accuracy  : 0.7707
  Precision : 0.7802
  Recall    : 0.7688
  F1-Score  : 0.7678

In [2]:
# =============================================================================
# Arabic Sentiment Analysis — BiLSTM + Multi-Head Attention (Fixed)
# =============================================================================

import warnings, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

TRAIN_PATH    = "/content/train.csv"
VAL_PATH      = "/content/val.csv"
TEST_PATH     = "/content/test.csv"
TEXT_COL      = "clean_text"
LABEL_COL     = "labels"

TOKENIZER_ID  = "aubmindlab/bert-base-arabertv2"
MAX_SEQ_LEN   = 128
NUM_LABELS    = 2

# Model hyperparameters
EMBED_DIM       = 300
LSTM_HIDDEN = 128
LSTM_LAYERS = 1
ATT_HEADS = 4
ATT_HEAD_DIM = 32
MLP_HIDDEN = 128
DROPOUT = 0.2

# Training
BATCH_SIZE    = 32
EPOCHS        = 4
LR            = 1e-4
WEIGHT_DECAY  = 1e-4
GRAD_CLIP     = 5.0
PATIENCE      = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device : {DEVICE}")


from transformers import AutoTokenizer
tokenizer  = AutoTokenizer.from_pretrained(TOKENIZER_ID)
VOCAB_SIZE = tokenizer.vocab_size
PAD_ID     = tokenizer.pad_token_id
print(f"  Tokenizer vocab : {VOCAB_SIZE:,} | PAD id : {PAD_ID}\n")


class ArabicSentimentDataset(Dataset):
    def __init__(self, df, max_len=MAX_SEQ_LEN):
        self.texts  = df[TEXT_COL].fillna("").astype(str).tolist()
        self.labels = df[LABEL_COL].astype(int).tolist()
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
            return_token_type_ids=False,
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long),
        }


def make_loader(df, shuffle):
    return DataLoader(
        ArabicSentimentDataset(df),
        batch_size=BATCH_SIZE, shuffle=shuffle,
        num_workers=0, pin_memory=DEVICE.type == "cuda"
    )


class HighwayLayer(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.transform = nn.Linear(dim, dim)
        self.gate      = nn.Linear(dim, dim)
        nn.init.constant_(self.gate.bias, -1.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        T = torch.sigmoid(self.gate(x))
        H = F.relu(self.transform(x))
        return T * H + (1.0 - T) * x


class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, input_dim: int, num_heads: int, head_dim: int):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = head_dim

        self.W = nn.Linear(input_dim, num_heads * head_dim)
        self.v = nn.Parameter(torch.randn(num_heads, head_dim))
        self.out_proj = nn.Linear(num_heads * head_dim, input_dim)
        self.dropout  = nn.Dropout(DROPOUT)
        nn.init.xavier_uniform_(self.v.unsqueeze(0))

    def forward(self, hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        B, T, D = hidden.shape

        # Project to all heads
        proj = torch.tanh(self.W(hidden)).view(B, T, self.num_heads, self.head_dim)  # B, T, H, d

        # Compute attention scores
        scores = torch.einsum("hd,bthd->bth", self.v, proj)  # B, T, H
        scores = scores.permute(0, 2, 1)                      # B, H, T

        # Mask padding positions
        mask_exp = mask.unsqueeze(1).expand_as(scores)       # B, H, T
        scores = scores.masked_fill(mask_exp == 0, -1e9)
        alpha  = F.softmax(scores, dim=-1)                  # B, H, T
        alpha  = self.dropout(alpha)

        # Weighted sum per head: B, H, T × B, T, H, d → B, H, d
        proj = proj.permute(0, 2, 1, 3)                     # B, H, T, d
        ctx_heads = torch.einsum("bht,bhtd->bhd", alpha, proj)  # B, H, d

        # Flatten heads and project
        ctx_flat = ctx_heads.reshape(B, self.num_heads * self.head_dim)  # B, H*d
        out = self.out_proj(ctx_flat)
        return self.dropout(out)


class BiLSTMAttentionClassifier(nn.Module):
    def __init__(self, vocab_size: int, num_labels: int = NUM_LABELS):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=PAD_ID)
        nn.init.uniform_(self.embedding.weight, -0.05, 0.05)
        with torch.no_grad():
            self.embedding.weight[PAD_ID].fill_(0.0)

        self.highway = HighwayLayer(EMBED_DIM)
        self.embed_dropout = nn.Dropout(DROPOUT)

        self.bilstm = nn.LSTM(
            input_size    = EMBED_DIM,
            hidden_size   = LSTM_HIDDEN,
            num_layers    = LSTM_LAYERS,
            bidirectional = True,
            batch_first   = True,
            dropout       = LSTM_DROPOUT if LSTM_LAYERS > 1 else 0.0,
        )
        lstm_out_dim = LSTM_HIDDEN * 2
        self.lstm_norm = nn.LayerNorm(lstm_out_dim)

        self.attention = MultiHeadAttentionPooling(
            input_dim = lstm_out_dim,
            num_heads = ATT_HEADS,
            head_dim  = ATT_HEAD_DIM,
        )

        self.residual_weight = nn.Parameter(torch.tensor(0.3))

        self.classifier = nn.Sequential(
            nn.Linear(lstm_out_dim, MLP_HIDDEN),
            nn.LayerNorm(MLP_HIDDEN),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(MLP_HIDDEN, MLP_HIDDEN // 2),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(MLP_HIDDEN // 2, num_labels),
        )

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor):
        emb = self.embedding(input_ids)
        emb = self.highway(emb)
        emb = self.embed_dropout(emb)

        lengths = attention_mask.sum(dim=1).cpu()
        packed  = nn.utils.rnn.pack_padded_sequence(
            emb, lengths, batch_first=True, enforce_sorted=False
        )
        lstm_out, _ = self.bilstm(packed)
        lstm_out, _ = nn.utils.rnn.pad_packed_sequence(
            lstm_out, batch_first=True, total_length=emb.size(1)
        )
        lstm_out = self.lstm_norm(lstm_out)

        attn_ctx = self.attention(lstm_out, attention_mask)

        mask_exp = attention_mask.unsqueeze(-1).float()
        masked   = lstm_out * mask_exp + (1 - mask_exp) * (-1e9)
        max_pool = masked.max(dim=1).values

        w      = torch.sigmoid(self.residual_weight)
        pooled = w * attn_ctx + (1.0 - w) * max_pool

        return self.classifier(pooled)


print("=" * 65); print("  📂  Loading Data"); print("=" * 65)
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)
print(f"  Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}")
print(f"\n  Labels:\n{train_df[LABEL_COL].value_counts()}\n")

train_loader = make_loader(train_df, shuffle=True)
val_loader   = make_loader(val_df,   shuffle=False)
test_loader  = make_loader(test_df,  shuffle=False)


model = BiLSTMAttentionClassifier(vocab_size=VOCAB_SIZE).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Trainable parameters : {total_params:,}\n")

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr         = LR,
    steps_per_epoch= len(train_loader),
    epochs         = EPOCHS,
    pct_start      = 0.2,
    anneal_strategy= "cos",
)

counts    = train_df[LABEL_COL].value_counts().sort_index()
weights   = torch.tensor([len(train_df) / (NUM_LABELS * c) for c in counts.values],
                         dtype=torch.float).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

def train_one_epoch(epoch):
    model.train()
    total_loss, preds_all, labels_all = 0.0, [], []
    for step, batch in enumerate(train_loader):
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        y    = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        logits = model(ids, mask)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        total_loss += loss.item()
        preds_all.extend(logits.argmax(-1).cpu().numpy())
        labels_all.extend(y.cpu().numpy())
        if (step + 1) % 100 == 0:
            print(f"    E{epoch+1} step {step+1}/{len(train_loader)} "
                  f"loss {loss.item():.4f} lr {scheduler.get_last_lr()[0]:.2e}")
    return (total_loss / len(train_loader),
            accuracy_score(labels_all, preds_all),
            f1_score(labels_all, preds_all, average="macro", zero_division=0))


@torch.no_grad()
def eval_loader(loader):
    model.eval()
    total_loss, preds_all, labels_all = 0.0, [], []
    for batch in loader:
        ids   = batch["input_ids"].to(DEVICE)
        mask  = batch["attention_mask"].to(DEVICE)
        y     = batch["label"].to(DEVICE)
        logits = model(ids, mask)
        total_loss += criterion(logits, y).item()
        preds_all.extend(logits.argmax(-1).cpu().numpy())
        labels_all.extend(y.cpu().numpy())
    return total_loss / len(loader), np.array(preds_all), np.array(labels_all)


def print_metrics(title, y_true, y_pred):
    names = ["Negative", "Positive"]
    print(f"\n{'─'*65}\n  {title}\n{'─'*65}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=names, zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")


print("=" * 65); print("  🚀  Training BiLSTM + Attention from Scratch"); print("=" * 65)

best_val_f1, best_state = 0.0, None
patience_counter        = 0
history                 = []

for epoch in range(EPOCHS):
    print(f"\n{'='*65}\n  EPOCH {epoch+1}/{EPOCHS}\n{'='*65}")
    tr_loss, tr_acc, tr_f1 = train_one_epoch(epoch)
    vl_loss, vl_preds, vl_labels = eval_loader(val_loader)
    vl_f1 = f1_score(vl_labels, vl_preds, average="macro", zero_division=0)
    history.append({"epoch": epoch+1, "train_loss": tr_loss, "train_f1": tr_f1,
                    "val_loss": vl_loss, "val_f1": vl_f1})
    print(f"\n  Train Loss {tr_loss:.4f} | Acc {tr_acc:.4f} | F1 {tr_f1:.4f}")
    print(f"  Val   Loss {vl_loss:.4f} | F1 {vl_f1:.4f}")

    if vl_f1 > best_val_f1:
        best_val_f1      = vl_f1
        best_state       = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        torch.save(best_state, "best_bilstm_attention.pt")
        print(f"  ✅ Best model saved (val F1 = {best_val_f1:.4f})")
    else:
        patience_counter += 1
        print(f"  ⏳ No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print(f"\n  Early stopping triggered after epoch {epoch+1}.")
            break


model.load_state_dict(best_state)
_, vp, vl = eval_loader(val_loader);  print_metrics("Validation Set", vl, vp)
_, tp, tl = eval_loader(test_loader); print_metrics("Test Set",       tl, tp)

print("\n" + pd.DataFrame(history).to_string(index=False, float_format="{:.4f}".format))
print("\n  Model saved to: best_bilstm_attention.pt")

✅ Device : cuda
  Tokenizer vocab : 64,000 | PAD id : 31

  📂  Loading Data
  Train 5,068 | Val 1,086 | Test 1,086

  Labels:
labels
1    2576
0    2492
Name: count, dtype: int64

  Trainable parameters : 19,929,019

  🚀  Training BiLSTM + Attention from Scratch

  EPOCH 1/4
    E1 step 100/159 loss 0.6921 lr 9.01e-05

  Train Loss 0.6960 | Acc 0.4992 | F1 0.4940
  Val   Loss 0.6899 | F1 0.3378
  ✅ Best model saved (val F1 = 0.3378)

  EPOCH 2/4
    E2 step 100/159 loss 0.6878 lr 8.41e-05

  Train Loss 0.6799 | Acc 0.5641 | F1 0.5641
  Val   Loss 0.6203 | F1 0.6723
  ✅ Best model saved (val F1 = 0.6723)

  EPOCH 3/4
    E3 step 100/159 loss 0.4502 lr 3.86e-05

  Train Loss 0.5127 | Acc 0.7664 | F1 0.7664
  Val   Loss 0.4896 | F1 0.7766
  ✅ Best model saved (val F1 = 0.7766)

  EPOCH 4/4
    E4 step 100/159 loss 0.3461 lr 3.17e-06

  Train Loss 0.3907 | Acc 0.8668 | F1 0.8668
  Val   Loss 0.4730 | F1 0.7942
  ✅ Best model saved (val F1 = 0.7942)

────────────────────────────────────────

In [3]:
# =============================================================================
# Arabic Sentiment Analysis — BiLSTM + Multi-Head Attention (Fixed)
# =============================================================================

import warnings, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

TRAIN_PATH    = "/content/train.csv"
VAL_PATH      = "/content/val.csv"
TEST_PATH     = "/content/test.csv"
TEXT_COL      = "clean_text"
LABEL_COL     = "labels"

TOKENIZER_ID  = "aubmindlab/bert-base-arabertv2"
MAX_SEQ_LEN   = 128
NUM_LABELS    = 2

# Model hyperparameters
EMBED_DIM       = 300
LSTM_HIDDEN = 128
LSTM_LAYERS = 1
ATT_HEADS = 4
ATT_HEAD_DIM = 32
MLP_HIDDEN = 128
DROPOUT = 0.5

# Training
BATCH_SIZE    = 32
EPOCHS        = 4
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
GRAD_CLIP     = 5.0
PATIENCE      = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device : {DEVICE}")


from transformers import AutoTokenizer
tokenizer  = AutoTokenizer.from_pretrained(TOKENIZER_ID)
VOCAB_SIZE = tokenizer.vocab_size
PAD_ID     = tokenizer.pad_token_id
print(f"  Tokenizer vocab : {VOCAB_SIZE:,} | PAD id : {PAD_ID}\n")


class ArabicSentimentDataset(Dataset):
    def __init__(self, df, max_len=MAX_SEQ_LEN):
        self.texts  = df[TEXT_COL].fillna("").astype(str).tolist()
        self.labels = df[LABEL_COL].astype(int).tolist()
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
            return_token_type_ids=False,
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long),
        }


def make_loader(df, shuffle):
    return DataLoader(
        ArabicSentimentDataset(df),
        batch_size=BATCH_SIZE, shuffle=shuffle,
        num_workers=0, pin_memory=DEVICE.type == "cuda"
    )


class HighwayLayer(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.transform = nn.Linear(dim, dim)
        self.gate      = nn.Linear(dim, dim)
        nn.init.constant_(self.gate.bias, -1.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        T = torch.sigmoid(self.gate(x))
        H = F.relu(self.transform(x))
        return T * H + (1.0 - T) * x


class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, input_dim: int, num_heads: int, head_dim: int):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = head_dim

        self.W = nn.Linear(input_dim, num_heads * head_dim)
        self.v = nn.Parameter(torch.randn(num_heads, head_dim))
        self.out_proj = nn.Linear(num_heads * head_dim, input_dim)
        self.dropout  = nn.Dropout(DROPOUT)
        nn.init.xavier_uniform_(self.v.unsqueeze(0))

    def forward(self, hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        B, T, D = hidden.shape

        # Project to all heads
        proj = torch.tanh(self.W(hidden)).view(B, T, self.num_heads, self.head_dim)  # B, T, H, d

        # Compute attention scores
        scores = torch.einsum("hd,bthd->bth", self.v, proj)  # B, T, H
        scores = scores.permute(0, 2, 1)                      # B, H, T

        # Mask padding positions
        mask_exp = mask.unsqueeze(1).expand_as(scores)       # B, H, T
        scores = scores.masked_fill(mask_exp == 0, -1e9)
        alpha  = F.softmax(scores, dim=-1)                  # B, H, T
        alpha  = self.dropout(alpha)

        # Weighted sum per head: B, H, T × B, T, H, d → B, H, d
        proj = proj.permute(0, 2, 1, 3)                     # B, H, T, d
        ctx_heads = torch.einsum("bht,bhtd->bhd", alpha, proj)  # B, H, d

        # Flatten heads and project
        ctx_flat = ctx_heads.reshape(B, self.num_heads * self.head_dim)  # B, H*d
        out = self.out_proj(ctx_flat)
        return self.dropout(out)


class BiLSTMAttentionClassifier(nn.Module):
    def __init__(self, vocab_size: int, num_labels: int = NUM_LABELS):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=PAD_ID)
        nn.init.uniform_(self.embedding.weight, -0.05, 0.05)
        with torch.no_grad():
            self.embedding.weight[PAD_ID].fill_(0.0)

        self.highway = HighwayLayer(EMBED_DIM)
        self.embed_dropout = nn.Dropout(DROPOUT)

        self.bilstm = nn.LSTM(
            input_size    = EMBED_DIM,
            hidden_size   = LSTM_HIDDEN,
            num_layers    = LSTM_LAYERS,
            bidirectional = True,
            batch_first   = True,
            dropout       = LSTM_DROPOUT if LSTM_LAYERS > 1 else 0.0,
        )
        lstm_out_dim = LSTM_HIDDEN * 2
        self.lstm_norm = nn.LayerNorm(lstm_out_dim)

        self.attention = MultiHeadAttentionPooling(
            input_dim = lstm_out_dim,
            num_heads = ATT_HEADS,
            head_dim  = ATT_HEAD_DIM,
        )

        self.residual_weight = nn.Parameter(torch.tensor(0.3))

        self.classifier = nn.Sequential(
            nn.Linear(lstm_out_dim, MLP_HIDDEN),
            nn.LayerNorm(MLP_HIDDEN),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(MLP_HIDDEN, MLP_HIDDEN // 2),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(MLP_HIDDEN // 2, num_labels),
        )

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor):
        emb = self.embedding(input_ids)
        emb = self.highway(emb)
        emb = self.embed_dropout(emb)

        lengths = attention_mask.sum(dim=1).cpu()
        packed  = nn.utils.rnn.pack_padded_sequence(
            emb, lengths, batch_first=True, enforce_sorted=False
        )
        lstm_out, _ = self.bilstm(packed)
        lstm_out, _ = nn.utils.rnn.pad_packed_sequence(
            lstm_out, batch_first=True, total_length=emb.size(1)
        )
        lstm_out = self.lstm_norm(lstm_out)

        attn_ctx = self.attention(lstm_out, attention_mask)

        mask_exp = attention_mask.unsqueeze(-1).float()
        masked   = lstm_out * mask_exp + (1 - mask_exp) * (-1e9)
        max_pool = masked.max(dim=1).values

        w      = torch.sigmoid(self.residual_weight)
        pooled = w * attn_ctx + (1.0 - w) * max_pool

        return self.classifier(pooled)


print("=" * 65); print("  📂  Loading Data"); print("=" * 65)
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)
print(f"  Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}")
print(f"\n  Labels:\n{train_df[LABEL_COL].value_counts()}\n")

train_loader = make_loader(train_df, shuffle=True)
val_loader   = make_loader(val_df,   shuffle=False)
test_loader  = make_loader(test_df,  shuffle=False)


model = BiLSTMAttentionClassifier(vocab_size=VOCAB_SIZE).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Trainable parameters : {total_params:,}\n")

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr         = LR,
    steps_per_epoch= len(train_loader),
    epochs         = EPOCHS,
    pct_start      = 0.2,
    anneal_strategy= "cos",
)

counts    = train_df[LABEL_COL].value_counts().sort_index()
weights   = torch.tensor([len(train_df) / (NUM_LABELS * c) for c in counts.values],
                         dtype=torch.float).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

def train_one_epoch(epoch):
    model.train()
    total_loss, preds_all, labels_all = 0.0, [], []
    for step, batch in enumerate(train_loader):
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        y    = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        logits = model(ids, mask)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        total_loss += loss.item()
        preds_all.extend(logits.argmax(-1).cpu().numpy())
        labels_all.extend(y.cpu().numpy())
        if (step + 1) % 100 == 0:
            print(f"    E{epoch+1} step {step+1}/{len(train_loader)} "
                  f"loss {loss.item():.4f} lr {scheduler.get_last_lr()[0]:.2e}")
    return (total_loss / len(train_loader),
            accuracy_score(labels_all, preds_all),
            f1_score(labels_all, preds_all, average="macro", zero_division=0))


@torch.no_grad()
def eval_loader(loader):
    model.eval()
    total_loss, preds_all, labels_all = 0.0, [], []
    for batch in loader:
        ids   = batch["input_ids"].to(DEVICE)
        mask  = batch["attention_mask"].to(DEVICE)
        y     = batch["label"].to(DEVICE)
        logits = model(ids, mask)
        total_loss += criterion(logits, y).item()
        preds_all.extend(logits.argmax(-1).cpu().numpy())
        labels_all.extend(y.cpu().numpy())
    return total_loss / len(loader), np.array(preds_all), np.array(labels_all)


def print_metrics(title, y_true, y_pred):
    names = ["Negative", "Positive"]
    print(f"\n{'─'*65}\n  {title}\n{'─'*65}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=names, zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")


print("=" * 65); print("  🚀  Training BiLSTM + Attention from Scratch"); print("=" * 65)

best_val_f1, best_state = 0.0, None
patience_counter        = 0
history                 = []

for epoch in range(EPOCHS):
    print(f"\n{'='*65}\n  EPOCH {epoch+1}/{EPOCHS}\n{'='*65}")
    tr_loss, tr_acc, tr_f1 = train_one_epoch(epoch)
    vl_loss, vl_preds, vl_labels = eval_loader(val_loader)
    vl_f1 = f1_score(vl_labels, vl_preds, average="macro", zero_division=0)
    history.append({"epoch": epoch+1, "train_loss": tr_loss, "train_f1": tr_f1,
                    "val_loss": vl_loss, "val_f1": vl_f1})
    print(f"\n  Train Loss {tr_loss:.4f} | Acc {tr_acc:.4f} | F1 {tr_f1:.4f}")
    print(f"  Val   Loss {vl_loss:.4f} | F1 {vl_f1:.4f}")

    if vl_f1 > best_val_f1:
        best_val_f1      = vl_f1
        best_state       = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        torch.save(best_state, "best_bilstm_attention.pt")
        print(f"  ✅ Best model saved (val F1 = {best_val_f1:.4f})")
    else:
        patience_counter += 1
        print(f"  ⏳ No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print(f"\n  Early stopping triggered after epoch {epoch+1}.")
            break


model.load_state_dict(best_state)
_, vp, vl = eval_loader(val_loader);  print_metrics("Validation Set", vl, vp)
_, tp, tl = eval_loader(test_loader); print_metrics("Test Set",       tl, tp)

print("\n" + pd.DataFrame(history).to_string(index=False, float_format="{:.4f}".format))
print("\n  Model saved to: best_bilstm_attention.pt")

✅ Device : cuda
  Tokenizer vocab : 64,000 | PAD id : 31

  📂  Loading Data
  Train 5,068 | Val 1,086 | Test 1,086

  Labels:
labels
1    2576
0    2492
Name: count, dtype: int64

  Trainable parameters : 19,929,019

  🚀  Training BiLSTM + Attention from Scratch

  EPOCH 1/4
    E1 step 100/159 loss 0.7011 lr 9.01e-04

  Train Loss 0.7024 | Acc 0.4901 | F1 0.4874
  Val   Loss 0.6920 | F1 0.3378
  ✅ Best model saved (val F1 = 0.3378)

  EPOCH 2/4
    E2 step 100/159 loss 0.6910 lr 8.41e-04

  Train Loss 0.6681 | Acc 0.5612 | F1 0.5595
  Val   Loss 0.5831 | F1 0.7062
  ✅ Best model saved (val F1 = 0.7062)

  EPOCH 3/4
    E3 step 100/159 loss 0.3157 lr 3.86e-04

  Train Loss 0.4160 | Acc 0.8469 | F1 0.8469
  Val   Loss 0.4329 | F1 0.8287
  ✅ Best model saved (val F1 = 0.8287)

  EPOCH 4/4
    E4 step 100/159 loss 0.2408 lr 3.17e-05

  Train Loss 0.2308 | Acc 0.9562 | F1 0.9562
  Val   Loss 0.4825 | F1 0.8203
  ⏳ No improvement (1/2)

──────────────────────────────────────────────────────